# Degrading CSTR simulator

This notebook introduces the synthetic plant that supplies the PICID experiment. It uses only the high-level API from `cstr_simulator.py`; all schedules, differential equations, integration, measurement noise, failure detection, and data export live in that module. Plot construction lives in `cstr_plotting.py`.

In [ ]:
from pathlib import Path

from IPython.display import Image, display
import pandas as pd

from cstr_plotting import (
    save_degradation_trace,
    save_fleet_overview,
    save_reactor_schema,
    save_threshold_detection_audit,
    threshold_detection_audit,
)
from cstr_simulator import (
    CSTR_FEATURE_COLUMNS,
    CSTRParameters,
    simulate_cstr_fleet,
)

WORKSHOP_ROOT = Path.cwd()
ASSET_DIR = WORKSHOP_ROOT / "assets"
DATA_DIR = WORKSHOP_ROOT / "data" / "cstr_simulator_seed7_10_units"
SEED = 7

## 1. The simulated plant

A continuous stirred-tank reactor (CSTR) receives reactant at concentration $C_{A,f}(t)$, temperature $T_f(t)$, and flow $q(t)$. Perfect mixing gives one reactor concentration $C_A(t)$ and temperature $T(t)$. An exothermic reaction consumes the reactant while a coolant jacket and PI controller follow a varying temperature setpoint $T_{set}(t)$.

This is a compact, literature-inspired teaching model rather than a calibrated representation of one chemical plant. Fast concentration, temperature, and controller responses are coupled to two slow latent health states: catalyst activity decreases while heat-transfer fouling resistance increases.

In [ ]:
schema_path = save_reactor_schema(ASSET_DIR / "cstr_reactor_schema.png")
display(Image(filename=str(schema_path)))

## 2. Coupled process and degradation equations

The reaction rate is

$$r(t)=a(t)\,k(T)\,C_A(t),$$

so declining catalyst activity $a(t)$ reduces conversion and reaction heat. Fouling resistance $R_f(t)$ reduces jacket performance through

$$U_{\mathrm{eff}}=\frac{U_{\mathrm{clean}}}{1+R_f},$$

The fast process balances are

$$\dot C_A=\frac{q}{V}(C_{A,f}-C_A)-r,$$

$$\dot T=\frac{q}{V}(T_f-T)+\beta r+K_{U,\mathrm{eff}}(T_c-T).$$

Reaction, deactivation, and fouling rates use Arrhenius temperature dependence. The two slow laws have the tutorial form

$$\dot a=-m_{a,i}\,k_d(T)\,g_a(q,C_{A,f})\,a^{1.1}\,\eta_{a,t},$$

$$\dot R_f=m_{f,i}\,k_f(T)\,g_f(q,r)\,\eta_{f,t}.$$

The mechanisms are distinct but coupled through shared temperature, flow, concentration, and reaction load. Unit susceptibilities and seeded positive process noise create different degradation paths. The signs enforce irreversible activity loss and fouling accumulation. Exact exponents and rates are synthetic workshop choices.

### What the model may observe

| Quantity | Role | Passed to PICID? |
|---|---|---|
| Reactor temperature $T$ | measured process response | yes |
| Coolant temperature $T_c$ | controller effort that redistributes health effects | yes |
| Feed flow $q$ | observed operating condition | yes |
| Feed concentration $C_{A,f}$ | observed operating condition | yes |
| Feed temperature $T_f$ | observed operating condition | yes |
| Temperature setpoint $T_{set}$ | observed controller context | yes |
| Catalyst activity $a$ | latent degradation ground truth | **no** |
| Fouling resistance $R_f$ | latent degradation ground truth | **no** |
| Outlet concentration and conversion | product-quality diagnostics | **no** |
| Reference capacity and pass/fail result | failure-test diagnostics | **no** |

The exported channels contain no direct product-quality, health, or failure-test measurement. Known operating inputs remain visible so their effects are context rather than hidden confounding. Degradation must be inferred from multivariate response and controller behavior over time.

## 3. Operational failure and RUL

Failure is not assigned by choosing a trajectory length or thresholding an exported sensor. At every health state, a hidden virtual test places the reactor at common reference conditions and searches for the maximum feed flow that simultaneously satisfies a minimum conversion, the reference temperature, and coolant actuator limits. The run stops after five consecutive tests place that capacity below the required production rate. RUL is the remaining simulated time until this first-passage event. Different deactivation/fouling susceptibilities create different health combinations at failure and therefore ragged lifetimes.

For the learnable tutorial configuration below, catalyst deactivation remains the dominant contributor while fouling stays active, and operating regimes last 80–120 samples instead of 20–44. This reduces cancellation and rapid context switching without exposing a latent health state or product-quality measurement.

In [ ]:
parameters = CSTRParameters(
    deactivation_rate_at_reference=0.009,
    fouling_rate_at_reference=0.003,
    regime_min_steps=80,
    regime_max_steps=120,
)
fleet = simulate_cstr_fleet(
    n_units=10,
    seed=SEED,
    parameters=parameters,
)
fleet.summary()

In [ ]:
fleet_path = save_fleet_overview(
    fleet,
    ASSET_DIR / "cstr_fleet_overview.png",
)
display(Image(filename=str(fleet_path)))

## 4. Follow one unit from degradation to failure

The upper plots are privileged simulator diagnostics: activity decreases, fouling accumulates, and the hidden reference capacity eventually crosses its requirement. Reactor temperature alone is deceptive because feedback follows a varying setpoint and redistributes health effects into coolant effort. Simultaneous changes in flow, feed concentration, feed temperature, and setpoint create response changes that are not faults.

In [ ]:
representative_unit = fleet.units["reactor_01"]
degradation_path = save_degradation_trace(
    representative_unit,
    ASSET_DIR / "cstr_degradation_trace.png",
)
display(Image(filename=str(degradation_path)))

## 5. Export only the PICID-facing data contract

`fleet.export_model_csvs()` is the persistent high-level boundary used by the experiment notebook. It writes one CSV per reactor with six observable/context features plus `rul`, and a manifest containing only file and shape metadata. Every latent, product-quality, and reference-test diagnostic is excluded. The CSVs below are reloaded to verify the exact disk representation passed to PICID's CSV/DataFrame loader.

In [ ]:
unit_csv_paths = fleet.export_model_csvs(DATA_DIR)
unit_frames = {
    unit_name: pd.read_csv(csv_path, dtype="float32")
    for unit_name, csv_path in unit_csv_paths.items()
}
csv_manifest = pd.read_csv(DATA_DIR / "manifest.csv")
assert tuple(unit_frames["reactor_01"].columns[:-1]) == CSTR_FEATURE_COLUMNS
assert "catalyst_activity" not in unit_frames["reactor_01"].columns
assert "fouling_resistance" not in unit_frames["reactor_01"].columns
assert "outlet_concentration" not in unit_frames["reactor_01"].columns
assert "reference_capacity" not in unit_frames["reactor_01"].columns

for unit_name, frame in unit_frames.items():
    print(
        f"{unit_name}: shape={frame.shape}, "
        f"RUL=[{frame['rul'].iloc[0]:.2f}, {frame['rul'].iloc[-1]:.2f}] min"
    )
print(f"Wrote {len(unit_csv_paths)} unit CSVs plus manifest to {DATA_DIR}")
display(csv_manifest)
unit_frames["reactor_01"].head()

In [ ]:
repeat = simulate_cstr_fleet(n_units=10, seed=SEED, parameters=parameters)
for unit_name in unit_frames:
    assert unit_frames[unit_name].equals(repeat.model_frames()[unit_name])
    trajectory = fleet.units[unit_name].trajectory
    assert (trajectory["catalyst_activity"].diff().dropna() <= 0.0).all()
    assert (trajectory["fouling_resistance"].diff().dropna() >= 0.0).all()
    assert bool(trajectory["is_failure"].iloc[-1])
    assert trajectory["reference_capacity"].iloc[-1] < parameters.reference_required_capacity
print("Seeded reproduction, irreversible health states, and hidden failure: verified")

## 6. Check that no scalar threshold solves the tutorial

For each raw channel, rolling mean/standard deviation/slope, and simple spectral summary of the response channels, the audit fits the best scalar cutoff on nine reactors and evaluates it on the held-out tenth reactor. Late life means the final 25% of a unit's lifetime. Balanced accuracy of 0.5 is chance and the dotted 0.70 line is the mean-score tutorial guardrail. The learnable configuration allows some average scalar signal, but every candidate must still fail on at least one held-out unit; this tests transferable univariate cutoffs, not every possible detector.

In [ ]:
threshold_audit = threshold_detection_audit(fleet)
threshold_path = save_threshold_detection_audit(
    threshold_audit,
    ASSET_DIR / "cstr_threshold_audit.png",
)
display(Image(filename=str(threshold_path)))

best_scalar_score = threshold_audit["mean_balanced_accuracy"].max()
best_worst_case_score = threshold_audit["min_balanced_accuracy"].max()
assert best_scalar_score < 0.70
assert best_worst_case_score < 0.50
threshold_audit.sort_values("mean_balanced_accuracy", ascending=False).head(8)

## 7. Literature basis: mechanisms versus tutorial choices

The implemented extension keeps the explanatory value of a CSTR while removing the concentration-threshold shortcut. It uses catalyst activity $a$ and heat-transfer fouling resistance $R_f$ as **two distinct latent states**:

$$r=a\,k(T)C_A, \qquad U_{\mathrm{eff}}=\frac{U_{\mathrm{clean}}}{1+R_f}.$$

They should not be called independent: temperature and load can influence both degradation rates. Catalyst activity loss reduces reaction rate and heat release; fouling reduces jacket heat removal. A feedback controller can attenuate the reactor-temperature signature and redistribute it into coolant effort, but it cannot be assumed to hide degradation completely. Feed concentration, flow, feed temperature, and setpoint should vary as known operating context so healthy and degraded sensor ranges overlap.

Outlet concentration and conversion are excluded from PICID. Failure and RUL are evaluated internally with the **virtual reference-capacity test** described above. This is a transparent benchmark construction inspired by catalyst testing under common conditions—not an established standardized CSTR test.

### What the literature does and does not establish

- Do and Weiland support coupled catalyst-activity/CSTR dynamics and deactivation-driven reactor failure ([1980](https://doi.org/10.1002/aic.690260618), [1981](https://doi.org/10.1016/0300-9467(81)80042-1)).
- Nikravesh, Farell, and Stanford combine catalyst deactivation, time-varying heat transfer used to represent fouling, feed-flow disturbances, and feedback in a nonisothermal CSTR simulation ([2000](https://doi.org/10.1016/S1385-8947(99)00108-4)).
- Fguiri, Marvillet, and Jeday support representing fouling as an increasing thermal resistance, although their industrial case is an external heat exchanger rather than a reactor jacket ([2021](https://doi.org/10.1016/j.applthermaleng.2021.116935)).
- Wan and Ye show that closed-loop action can redistribute degradation signatures across manipulated and process variables; their mechanism is sensor degradation, so this supports only the general feedback-masking claim ([2012](https://doi.org/10.1016/j.jprocont.2011.10.013)).
- Birtill, Deeley, and Bailey compare catalyst activity under common preset conditions, motivating reference-condition evaluation but not a universal hidden stress test ([2021](https://doi.org/10.1007/s11244-021-01488-z)).

The exact stochastic laws, parameter distributions, disturbance schedule, and failure limit remain synthetic design choices. The full evidence matrix and its cautions are kept in [`CSTR_DEGRADATION_LITERATURE.md`](CSTR_DEGRADATION_LITERATURE.md).

## Simulation Interpretation

The executable simulator's parameters are tuned for a compact workshop—not identified from a particular reactor. The threshold audit demonstrates one seeded tutorial fleet; it does not prove that every scalar feature or every future parameterization is uninformative. The simulator can test interfaces, leakage boundaries, windowing, and monitoring logic, but it must not be presented as evidence that a trained model will transfer to an industrial CSTR.

The existing model structure remains motivated by the classic catalyst-deactivation CSTR studies already cited here: Do and Weiland, *AIChE Journal* (1980), DOI [10.1002/aic.690260618](https://doi.org/10.1002/aic.690260618), and *Chemical Engineering Journal* (1981), DOI [10.1016/0300-9467(81)80042-1](https://doi.org/10.1016/0300-9467(81)80042-1). The additional literature above extends rather than replaces that basis.